# 04_deep_learning

Train the PyTorch fully-connected deep network.

---

*DiaPredict AI — Explainable Diabetes Risk Prediction System.* *Educational / research prototype, not a medical diagnosis.*

## Objectives
- Build the PyTorch MLP
- Train with a validation loop, Adam and early stopping
- Inspect the training/validation curves and test-set metrics

In [ ]:
import sys
from pathlib import Path

# Make the project root importable no matter where the notebook runs.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Root:", PROJECT_ROOT)


### 1. Prepare data

In [ ]:
from sklearn.model_selection import train_test_split
from backend.app.preprocessing import load_raw_dataset, make_train_test

raw = load_raw_dataset()
X_train, y_train, X_test, y_test, scaler, medians = make_train_test(raw)
X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, test_size=0.20, random_state=42, stratify=y_train)
print("Train:", X_tr.shape, "Val:", X_va.shape, "Test:", X_test.shape)

### 2. Build the network

In [ ]:
import torch, torch.nn as nn
from backend.app.model_loader import MLP

net = MLP(input_dim=X_tr.shape[1])
print(net)
print("\nTotal parameters:", sum(p.numel() for p in net.parameters()))

### 3. Training loop with early stopping

In [ ]:
from scripts.train_dl import Trainer

trainer = Trainer(input_dim=X_tr.shape[1], lr=1e-3, epochs=200, patience=25, batch_size=64)
trainer.fit(X_tr, y_tr, X_va, y_va)

### 4. Training / validation curves

In [ ]:
h = trainer.history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(h["train_loss"], label="train loss", color="#60a5fa")
axes[0].plot(h["val_loss"], label="val loss", color="#f87171")
axes[0].set_title("Loss curves"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(h["val_auc"], label="val ROC-AUC", color="#34d399")
axes[1].set_title("Validation AUC"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

### 5. Test-set evaluation

In [ ]:
metrics = trainer.evaluate(X_test, y_test)
for k, v in metrics.items():
    print(f"{k:20s} {v}")

## Notes

The network uses two hidden layers (24 → 16 units) with ReLU activations, dropout regularisation and a sigmoid output. Early stopping picks the checkpoint with the best validation AUC, so the final weights are not the last epoch's — this guards against overfitting on this small dataset.